In [ ]:
from pathlib import Path
import os
from os import listdir as ls
import arviz as az
import pickle
import pandas as pd
import pycountry
import pycountry_convert as pc
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import h5py  # Make sure h5py initialises properly before pandas ruins it
from IPython.display import display, Markdown

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, plot_prior_multipost, add_cont_to_world_geodf, plot_continent_grouping
from emu_renewal.utils import get_countries_by_continent

set_matplotlib_formats("svg")

# Purpose
This document presents the calibration to data
for each country, analysis and target indicator for fitting.

# Country grouping
For this document and the following supplemental files,
we grouped countries according to the following continent groupings.

In [ ]:
#| fig-align: center
world = get_world_shp()
add_cont_to_world_geodf(world)
plt.style.use("default")
plot_continent_grouping(world)

# Results by continent and country

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
for cont, countries in countries_by_cont.items():
    display(Markdown(f"## {pc.convert_continent_code_to_continent_name(cont)}"))
    for c in countries[:2]:
        country_name = pycountry.countries.lookup(c).name
        display(Markdown(f"### {country_name}"))
        analyses = {d.name: Path(d.path) for d in os.scandir(job_path / c) if d.is_dir()}
        
        no_mob_path = analyses["no_mob"]
    
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analyses.items()}
        targs = load_targets(no_mob_path)
        display(Markdown(f"#### Fit to data, {country_name}"))
        display(plot_multianalysis_fit(c, targs, spaghs))
            
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
        var_names = ["start"] + [v[5:] for v in targs.keys() if v.startswith("prop_")]
        display(Markdown(f"#### Prior-posterior comparisons, {country_name}"))
        display(plot_prior_multipost(c, var_names, priors, idatas, 4))